# 02. SECOM Preprocessing

이 노트북은 **EDA용 cleaned dataset**을 만든다. Timestamp와 원본 `-1/1` Target은 보존하고, 수치 Feature에서 constant·동일·고결측 열을 제거한 뒤 median으로 결측치를 채운다.

> **Leakage 주의:** 여기서 전체 데이터로 계산한 median은 EDA용이다. 모델링 단계에서는 먼저 train/test를 나누고, train 데이터로만 학습되는 imputer와 scaler를 Pipeline 안에서 다시 적용해야 한다.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError(f'Project root could not be resolved from {cwd}')
os.environ['MPLCONFIGDIR'] = str(PROJECT_ROOT / '.cache' / 'matplotlib')
sys.path.insert(0, str(PROJECT_ROOT))

import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display
from src.load_data import load_secom_dataset, split_secom_data

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_rows', 40)
print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Projects\semiconductor-secom-yield-analysis


## 1. Load and preserve timestamp/target

`timestamp`는 원본 문자열 Series로 보존하되 수치 모델 입력에서는 제외한다. Target `class`도 원본 `-1/1`을 유지한다. 별도의 `target_binary`만 편의상 만들며 저장된 원본 Target을 덮어쓰지 않는다.

In [2]:
dataset = load_secom_dataset()
X_raw, y_frame = split_secom_data(dataset)
X_raw = X_raw.copy()
y_original = y_frame['class'].copy()
timestamps = X_raw['timestamp'].copy()
X_numeric = X_raw.drop(columns='timestamp').copy()
target_binary = y_original.map({-1: 0, 1: 1}).rename('class_binary')
assert y_original.equals(y_frame['class'])
assert set(target_binary.unique()) == {0, 1}
print(f'Raw X shape: {X_raw.shape}')
print(f'Numeric model-input candidates: {X_numeric.shape}')
print(f'Timestamp preserved: {timestamps.shape}, dtype={timestamps.dtype}')
print(f'Original target values: {sorted(y_original.unique())}')

Raw X shape: (1567, 591)
Numeric model-input candidates: (1567, 590)
Timestamp preserved: (1567,), dtype=str
Original target values: [np.int64(-1), np.int64(1)]


## 2. Exact duplicate columns and constant features

동일 column 그룹에서는 원래 순서의 첫 열을 대표로 정한다. 실제 데이터의 동일 그룹은 모두 constant Feature로도 판정되므로, 중복 104개를 먼저 제거하고 남은 대표 constant 및 기타 constant 12개를 제거한다. 결과적으로 Step 1에서 확인한 constant Feature 116개는 모두 최종 입력에서 제외된다.

In [3]:
original_nunique = X_numeric.nunique(dropna=True)
original_constant_features = original_nunique[original_nunique <= 1].index.tolist()
hash_buckets = {}
for column in X_numeric.columns:
    values = pd.util.hash_pandas_object(X_numeric[column], index=False).values.tobytes()
    hash_buckets.setdefault(hashlib.sha256(values).hexdigest(), []).append(column)
duplicate_groups = []
for candidates in hash_buckets.values():
    remaining = list(candidates)
    while remaining:
        representative = remaining.pop(0)
        group = [representative]
        for candidate in remaining.copy():
            if X_numeric[representative].equals(X_numeric[candidate]):
                group.append(candidate)
                remaining.remove(candidate)
        if len(group) > 1:
            duplicate_groups.append(group)
duplicate_representatives = [group[0] for group in duplicate_groups]
duplicate_columns_removed = [column for group in duplicate_groups for column in group[1:]]
X_deduplicated = X_numeric.drop(columns=duplicate_columns_removed)
remaining_constant_features = X_deduplicated.columns[X_deduplicated.nunique(dropna=True) <= 1].tolist()
X_quality_base = X_deduplicated.drop(columns=remaining_constant_features)
assert len(original_constant_features) == 116
assert not (X_quality_base.nunique(dropna=True) <= 1).any()
duplicate_audit = pd.DataFrame({
    'representative': duplicate_representatives,
    'removed_columns': [', '.join(group[1:]) for group in duplicate_groups],
    'group_size': [len(group) for group in duplicate_groups],
})
print(f'Original constant features identified: {len(original_constant_features)}')
print(f'Exact duplicate groups: {len(duplicate_groups)}')
print(f'Duplicate columns removed first: {len(duplicate_columns_removed)}')
print(f'Additional constant columns removed: {len(remaining_constant_features)}')
print(f'Features after duplicate/constant removal: {X_quality_base.shape[1]}')
display(duplicate_audit)

Original constant features identified: 116
Exact duplicate groups: 7
Duplicate columns removed first: 104
Additional constant columns removed: 12
Features after duplicate/constant removal: 474


,representative,removed_columns,group_size
0,Attribute 14,"Attribute 150, Attribute 285, Attribute 423",4
1,Attribute 53,"Attribute 180, Attribute 187, Attribute 190, A...",10
2,Attribute 98,"Attribute 227, Attribute 230, Attribute 231, A...",41
3,Attribute 142,"Attribute 277, Attribute 415",3
4,Attribute 179,"Attribute 314, Attribute 315, Attribute 450, A...",5
5,Attribute 191,"Attribute 192, Attribute 193, Attribute 194, A...",15
6,Attribute 257,"Attribute 258, Attribute 259, Attribute 260, A...",33


## 3. Select a high-missing threshold

40%, 50%, 60% 초과 기준을 실제 분포로 비교한다. 40% 기준은 50%보다 4개만 더 제거하면서 최대 잔존 결측률을 45.63%에서 17.42%로 낮춘다. 뚜렷한 분포 간격을 활용할 수 있으므로 **40% 초과 제거**를 선택한다.

In [4]:
missing_rates = X_quality_base.isna().mean()
threshold_rows = []
for threshold in [0.40, 0.50, 0.60]:
    removed = int((missing_rates > threshold).sum())
    remaining_rates = missing_rates[missing_rates <= threshold]
    threshold_rows.append({
        'threshold': threshold,
        'removed_features': removed,
        'remaining_features': int(len(remaining_rates)),
        'max_remaining_missing_rate': float(remaining_rates.max()),
    })
threshold_comparison = pd.DataFrame(threshold_rows)
display(threshold_comparison)
SELECTED_THRESHOLD = 0.40
high_missing_features = missing_rates[missing_rates > SELECTED_THRESHOLD].sort_values(ascending=False).index.tolist()
X_retained = X_quality_base.drop(columns=high_missing_features)
print(f'Selected threshold: > {SELECTED_THRESHOLD:.0%}')
print(f'High-missing features removed: {len(high_missing_features)}')
print(high_missing_features)

,threshold,removed_features,remaining_features,max_remaining_missing_rate
0,0.4,32,442,0.174218
1,0.5,28,446,0.456286
2,0.6,24,450,0.506701


Selected threshold: > 40%
High-missing features removed: 32
['Attribute 158', 'Attribute 293', 'Attribute 294', 'Attribute 159', 'Attribute 221', 'Attribute 86', 'Attribute 493', 'Attribute 359', 'Attribute 246', 'Attribute 245', 'Attribute 247', 'Attribute 112', 'Attribute 383', 'Attribute 384', 'Attribute 111', 'Attribute 110', 'Attribute 385', 'Attribute 519', 'Attribute 518', 'Attribute 517', 'Attribute 581', 'Attribute 580', 'Attribute 582', 'Attribute 579', 'Attribute 347', 'Attribute 346', 'Attribute 73', 'Attribute 74', 'Attribute 248', 'Attribute 113', 'Attribute 386', 'Attribute 520']


## 4. Median imputation for the EDA dataset

남은 수치 Feature의 결측치는 각 열의 median으로 채운다. 이 값은 EDA용이며 모델 평가에 재사용하지 않는다. 모델링에서는 split 이후 train-only Pipeline으로 다시 계산한다.

In [5]:
missing_before_imputation = int(X_retained.isna().sum().sum())
feature_medians = X_retained.median(axis=0)
if feature_medians.isna().any():
    raise ValueError('At least one retained feature has no usable median.')
X_cleaned = X_retained.fillna(feature_medians)
remaining_missing = int(X_cleaned.isna().sum().sum())
assert remaining_missing == 0
assert timestamps.equals(X_raw['timestamp'])
assert y_original.equals(y_frame['class'])
cleaned_dataset = pd.concat([
    timestamps.rename('timestamp'), X_cleaned, y_original.rename('class')
], axis=1)
processed_path = PROCESSED_DIR / 'secom_cleaned_eda.csv'
cleaned_dataset.to_csv(processed_path, index=False)
print(f'Missing cells before median imputation: {missing_before_imputation:,}')
print(f'Remaining missing cells: {remaining_missing:,}')
print(f'Numeric cleaned shape: {X_cleaned.shape}')
print(f'Saved dataset shape (timestamp + numeric + target): {cleaned_dataset.shape}')
print(f'Saved: {processed_path}')

Missing cells before median imputation: 8,008
Remaining missing cells: 0
Numeric cleaned shape: (1567, 442)
Saved dataset shape (timestamp + numeric + target): (1567, 444)
Saved: C:\Projects\semiconductor-secom-yield-analysis\data\processed\secom_cleaned_eda.csv


## 5. Before/after audit and saved summary

In [6]:
audit_table = pd.DataFrame([
    {'stage': 'Raw numeric features', 'feature_count': X_numeric.shape[1], 'missing_cells': int(X_numeric.isna().sum().sum())},
    {'stage': 'After duplicate columns', 'feature_count': X_deduplicated.shape[1], 'missing_cells': int(X_deduplicated.isna().sum().sum())},
    {'stage': 'After constant features', 'feature_count': X_quality_base.shape[1], 'missing_cells': int(X_quality_base.isna().sum().sum())},
    {'stage': 'After high-missing features', 'feature_count': X_retained.shape[1], 'missing_cells': missing_before_imputation},
    {'stage': 'After EDA median imputation', 'feature_count': X_cleaned.shape[1], 'missing_cells': remaining_missing},
])
display(audit_table)
summary = {
    'purpose': 'EDA-only cleaned dataset; refit preprocessing after train/test split for modeling',
    'raw_x_shape_including_timestamp': list(X_raw.shape),
    'raw_numeric_shape': list(X_numeric.shape),
    'timestamp_preserved': True,
    'target_values_preserved': sorted(int(v) for v in y_original.unique()),
    'original_constant_feature_count': len(original_constant_features),
    'duplicate_group_count': len(duplicate_groups),
    'duplicate_columns_removed': len(duplicate_columns_removed),
    'duplicate_columns_removed_list': duplicate_columns_removed,
    'additional_constant_columns_removed': len(remaining_constant_features),
    'remaining_constant_columns_removed_list': remaining_constant_features,
    'threshold_comparison': threshold_comparison.to_dict(orient='records'),
    'selected_missing_threshold': SELECTED_THRESHOLD,
    'high_missing_features_removed': len(high_missing_features),
    'high_missing_features_removed_list': high_missing_features,
    'missing_before_imputation': missing_before_imputation,
    'remaining_missing': remaining_missing,
    'final_numeric_shape': list(X_cleaned.shape),
    'saved_dataset_shape': list(cleaned_dataset.shape),
    'processed_dataset': str(processed_path),
}
summary_path = REPORTS_DIR / 'preprocessing_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(summary, indent=2, ensure_ascii=False))
print(f'Saved: {summary_path}')

,stage,feature_count,missing_cells
0,Raw numeric features,590,41951
1,After duplicate columns,486,41225
2,After constant features,474,41136
3,After high-missing features,442,8008
4,After EDA median imputation,442,0


{
  "purpose": "EDA-only cleaned dataset; refit preprocessing after train/test split for modeling",
  "raw_x_shape_including_timestamp": [
    1567,
    591
  ],
  "raw_numeric_shape": [
    1567,
    590
  ],
  "timestamp_preserved": true,
  "target_values_preserved": [
    -1,
    1
  ],
  "original_constant_feature_count": 116,
  "duplicate_group_count": 7,
  "duplicate_columns_removed": 104,
  "duplicate_columns_removed_list": [
    "Attribute 150",
    "Attribute 285",
    "Attribute 423",
    "Attribute 180",
    "Attribute 187",
    "Attribute 190",
    "Attribute 316",
    "Attribute 323",
    "Attribute 326",
    "Attribute 452",
    "Attribute 459",
    "Attribute 462",
    "Attribute 227",
    "Attribute 230",
    "Attribute 231",
    "Attribute 232",
    "Attribute 233",
    "Attribute 234",
    "Attribute 235",
    "Attribute 236",
    "Attribute 237",
    "Attribute 238",
    "Attribute 241",
    "Attribute 242",
    "Attribute 243",
    "Attribute 244",
    "Attribute 36

## Scope boundary

Feature selection, PCA, SMOTE, 모델 학습, scaling 및 SHAP은 수행하지 않았다. 후속 모델링에서는 원본 Feature로부터 split을 먼저 수행하고 preprocessing Pipeline을 train fold에만 fit한다.